In [1]:
import pandas as pd
import numpy as np

# ==============================
# 1. Read data
# ==============================
df = pd.read_csv("tmin.csv", parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)

# ==============================
# 2. Define reference period
# ==============================
ref_start = "2000-01-01"
ref_end   = "2016-12-31"

ref = df[(df["date"] >= ref_start) & (df["date"] <= ref_end)]

# Basic checks, very important
assert ref["gmfd_tmin"].isna().sum() == 0, "GMFD reference period contains missing values"
assert ref["era5_tmin"].isna().sum() == 0, "ERA5 reference period contains missing values"

# ==============================
# 3. QDM additive function
# ==============================
def qdm_additive(x, model_ref, obs_ref):
    """
    Additive Quantile Delta Mapping
    x          : ERA5 value (any time)
    model_ref : ERA5 values during reference period
    obs_ref   : GMFD values during reference period
    """
    # Quantile, empirical CDF
    q = np.sum(model_ref <= x) / len(model_ref)
    q = np.clip(q, 0.001, 0.999)

    # Corresponding quantile values
    model_q = np.quantile(model_ref, q)
    obs_q = np.quantile(obs_ref, q)

    # Delta and correction
    delta = obs_q - model_q
    return x + delta

# ==============================
# 4. Apply QDM
# ==============================
model_ref = ref["era5_tmin"].values
obs_ref = ref["gmfd_tmin"].values

df["era5_tmin_qdm"] = df["era5_tmin"].apply(
    lambda x: qdm_additive(x, model_ref, obs_ref)
)

# ==============================
# 5. Save results
# ==============================
df.to_csv("tmin_qdm.csv", index=False)

print("QDM bias correction finished.")
print("Reference period: 2000–2016")
print("Corrected period: 2000–2024")

/Users/liuqi/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


QDM bias correction finished.
Reference period: 2000–2016
Corrected period: 2000–2024
